# Week 1 - CloudSEN12 Dataset Exploration & Configuration

**Project:** Cloud Detection - Comparing Models for Cloud Masking

**Objective:** Complete exploration of CloudSEN12 dataset and establishment of experimental setup

## Week 1 Agenda
1. Load and analyze CloudSEN12 dataset
2. Understand band structure and metadata
3. Visualize sample images and cloud masks
4. Analyze class distribution
5. Define subset strategy and train/val/test split
6. Select input bands for models
7. Define preprocessing pipeline
8. Save configuration for reproducibility

## Environment Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import json
from collections import defaultdict

# Create output directories
os.makedirs('../outputs/figures', exist_ok=True)

print("Environment Setup:")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Working directory: {os.getcwd()}")
print("✓ All libraries imported successfully")

## Step 1: Load CloudSEN12 Dataset

In [ ]:
import tacoreader.v1 as tacoreader

# Load CloudSEN12 L2A dataset
dataset = tacoreader.load("tacofoundation:cloudsen12-l2a")

print("\n=== CloudSEN12 Dataset Loaded ===")
print(f"Type: {type(dataset).__name__}")
print(f"Total samples: {len(dataset):,}")
print(f"Shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")
print("✓ Dataset ready for exploration")

## Step 2: Analyze Dataset Structure

In [ ]:
# Analyze image and mask structure
band_info = defaultdict(list)
image_shapes = []
target_shapes = []

print("Analyzing first 20 samples...\n")

for i in range(min(20, len(dataset))):
    sample = dataset.read(i)
    image_path = sample.read(0)
    target_path = sample.read(1)
    
    with rasterio.open(image_path) as src:
        image = src.read()
        image_shapes.append(image.shape)
        band_info[image.shape[0]].append((i, image.dtype, float(image.min()), float(image.max())))
    
    with rasterio.open(target_path) as src:
        target = src.read()
        target_shapes.append(target.shape)

print("=== Dataset Statistics ===")
print(f"Image shapes found: {set(image_shapes)}")
print(f"Target shapes found: {set(target_shapes)}")

print(f"\n=== Image Band Information ===")
for n_bands in sorted(band_info.keys()):
    print(f"\nImages with {n_bands} bands:")
    for idx, dtype, min_val, max_val in band_info[n_bands][:2]:
        print(f"  Sample {idx}: dtype={dtype}, range=[{min_val:.0f}, {max_val:.0f}]")

print(f"\n=== Target Mask Information ===")
for idx in range(min(3, len(dataset))):
    sample = dataset.read(idx)
    target_path = sample.read(1)
    with rasterio.open(target_path) as src:
        target = src.read()
        unique_vals = sorted(np.unique(target).tolist())
        print(f"Sample {idx}: shape={target.shape}, unique values={unique_vals}, dtype={target.dtype}")

## Step 3: Sentinel-2 Band Information

In [ ]:
# Sentinel-2 L2A band information
sentinel2_bands = {
    0: {'name': 'B1', 'description': 'Coastal aerosol', 'resolution': '60m'},
    1: {'name': 'B2', 'description': 'Blue', 'resolution': '10m'},
    2: {'name': 'B3', 'description': 'Green', 'resolution': '10m'},
    3: {'name': 'B4', 'description': 'Red', 'resolution': '10m'},
    4: {'name': 'B5', 'description': 'Vegetation Red Edge', 'resolution': '20m'},
    5: {'name': 'B6', 'description': 'Vegetation Red Edge', 'resolution': '20m'},
    6: {'name': 'B7', 'description': 'Vegetation Red Edge', 'resolution': '20m'},
    7: {'name': 'B8', 'description': 'NIR', 'resolution': '10m'},
    8: {'name': 'B8A', 'description': 'Vegetation Red Edge', 'resolution': '20m'},
    9: {'name': 'B11', 'description': 'SWIR', 'resolution': '20m'},
    10: {'name': 'B12', 'description': 'SWIR', 'resolution': '20m'},
    11: {'name': 'SCL', 'description': 'Scene Classification', 'resolution': '20m'}
}

print("=== Sentinel-2 Bands (CloudSEN12 L2A) ===")
print(f"Total bands available: {len(sentinel2_bands)}\n")
for idx, info in sentinel2_bands.items():
    print(f"Index {idx:2d} - {info['name']:4s} ({info['resolution']:4s}): {info['description']}")

## Step 4: Visualize Samples

In [ ]:
# Cloud mask class labels
class_labels = {
    0: 'Clear sky',
    1: 'Thin cloud',
    2: 'Thick cloud',
    3: 'Cloud shadow'
}

# Visualize samples
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('CloudSEN12 Dataset - Sample Images and Cloud Masks', fontsize=16, y=0.995)

for idx in range(3):
    sample = dataset.read(idx)
    image_path = sample.read(0)
    target_path = sample.read(1)
    
    with rasterio.open(image_path) as src:
        image = src.read()
    
    with rasterio.open(target_path) as src:
        target = src.read()
    
    # Extract RGB bands (indices 3, 2, 1 = R, G, B)
    if image.shape[0] >= 4:
        rgb = np.stack([image[3], image[2], image[1]], axis=0)
        rgb = np.clip(rgb / 3000.0, 0, 1)
    else:
        rgb = np.clip(image[:3] / 3000.0, 0, 1)
    
    # Plot 1: RGB Image
    ax = axes[idx, 0]
    ax.imshow(np.transpose(rgb, (1, 2, 0)))
    ax.set_title(f'Sample {idx} - RGB Image', fontsize=12)
    ax.axis('off')
    
    # Plot 2: Cloud Mask (grayscale)
    ax = axes[idx, 1]
    ax.imshow(target.squeeze(), cmap='gray')
    ax.set_title(f'Sample {idx} - Cloud Mask', fontsize=12)
    ax.axis('off')
    
    # Plot 3: Cloud Classes (colored)
    ax = axes[idx, 2]
    mask_colored = np.zeros((target.shape[1], target.shape[2], 3))
    unique_vals = np.unique(target)
    colors = np.array([[0.2, 0.8, 0.2],   # Clear - green
                      [1.0, 1.0, 0.0],    # Thin - yellow
                      [1.0, 0.0, 0.0],    # Thick - red
                      [0.0, 0.0, 1.0]])   # Shadow - blue
    
    for val in unique_vals:
        if val < 4:
            mask_colored[target.squeeze() == val] = colors[int(val)]
    
    ax.imshow(mask_colored)
    ax.set_title(f'Sample {idx} - Cloud Classes', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/figures/01_dataset_samples.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to outputs/figures/01_dataset_samples.png")

## Step 5: Class Distribution Analysis

In [ ]:
# Analyze class distribution
class_distribution = defaultdict(int)
total_pixels = 0

print("Analyzing class distribution (first 50 samples)...\n")

for i in range(min(50, len(dataset))):
    sample = dataset.read(i)
    target_path = sample.read(1)
    
    with rasterio.open(target_path) as src:
        target = src.read()
        unique, counts = np.unique(target, return_counts=True)
        for val, count in zip(unique, counts):
            class_distribution[int(val)] += count
        total_pixels += target.size

print("=== Cloud Mask Class Distribution ===")
print(f"Total pixels analyzed: {total_pixels:,}\n")
print("Class distribution:")

for class_val in sorted(class_distribution.keys()):
    count = class_distribution[class_val]
    percentage = (count / total_pixels) * 100
    label = class_labels.get(class_val, f'Unknown ({class_val})')
    print(f"Class {class_val} ({label:20s}): {count:12,d} pixels ({percentage:6.2f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
classes = sorted(class_distribution.keys())
labels = [class_labels.get(c, f'Unknown ({c})') for c in classes]
counts = [class_distribution[c] for c in classes]

colors_bar = ['#2ecc71', '#f39c12', '#e74c3c', '#3498db']
ax.bar(labels, counts, color=colors_bar[:len(labels)])
ax.set_ylabel('Number of pixels')
ax.set_title('Cloud Mask Class Distribution (first 50 samples)')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(counts):
    ax.text(i, v, f'{v/1e6:.1f}M', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../outputs/figures/02_class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved to outputs/figures/02_class_distribution.png")

## Step 6: Define Subset & Train/Val/Test Split

In [ ]:
# Define subset strategy
full_size = len(dataset)
print(f"Full CloudSEN12 dataset size: {full_size:,} samples")

# Use 10% for Week 1
subset_fraction = 0.10
subset_size = max(100, int(full_size * subset_fraction))

print(f"\nSubset Strategy:")
print(f"  Subset size: {subset_size:,} samples ({subset_fraction*100:.1f}% of {full_size:,})")
print(f"  Rationale: Manageable exploration while remaining representative")

# Define split percentages
train_fraction = 0.60
val_fraction = 0.20
test_fraction = 0.20

print(f"\nTrain/Val/Test Split Fractions:")
print(f"  Training:   {train_fraction*100:.0f}%")
print(f"  Validation: {val_fraction*100:.0f}%")
print(f"  Test:       {test_fraction*100:.0f}%")

# Calculate sizes
train_size = int(subset_size * train_fraction)
val_size = int(subset_size * val_fraction)
test_size = subset_size - train_size - val_size

train_indices = list(range(0, train_size))
val_indices = list(range(train_size, train_size + val_size))
test_indices = list(range(train_size + val_size, subset_size))

print(f"\nActual sample counts:")
print(f"  Training:   {len(train_indices):,d} samples")
print(f"  Validation: {len(val_indices):,d} samples")
print(f"  Test:       {len(test_indices):,d} samples")
print(f"  Total:      {len(train_indices) + len(val_indices) + len(test_indices):,d} samples")

## Step 7: Input Band Selection

In [ ]:
print("=== Input Band Selection for Models ===")

print("\nOption 1 - RGB (3 bands):")
print("  Bands: B4 (Red), B3 (Green), B2 (Blue)")
print("  Pros: Simple, fast")
print("  Cons: Limited spectral information")

print("\nOption 2 - RGB + NIR (4 bands):")
print("  Bands: B4 (Red), B3 (Green), B2 (Blue), B8 (NIR)")
print("  Pros: Good cloud discrimination, standard in remote sensing")
print("  Cons: Slightly more computation")

print("\nOption 3 - Full 10m multispectral (4 bands):")
print("  Bands: B2, B3, B4, B8 (all at 10m)")
print("  Same as Option 2")

print("\nOption 4 - Rich multispectral (7 bands):")
print("  Bands: B2, B3, B4, B5, B8, B11, B12")
print("  Pros: Maximum spectral information")
print("  Cons: Mixed resolutions, higher computation")

print("\n" + "="*70)
print("✓ SELECTED: Option 2 - RGB + NIR (4 bands)")
print("="*70)
print("\nReason: Excellent balance between spectral information and")
print("        computational efficiency. NDVI and visible bands provide")
print("        strong discrimination for cloud detection.")

selected_band_indices = [3, 2, 1, 7]  # Red, Green, Blue, NIR
selected_band_names = ['Red (B4)', 'Green (B3)', 'Blue (B2)', 'NIR (B8)']

print(f"\nFinal selection:")
for idx, name in zip(selected_band_indices, selected_band_names):
    print(f"  Index {idx}: {name}")

## Step 8: Preprocessing & Normalization Strategy

In [ ]:
print("\n=== Preprocessing Pipeline ===")

preprocessing_steps = [
    {'step': 1, 'name': 'Extract bands', 'description': 'Red, Green, Blue, NIR (indices 3,2,1,7)', 'output': '(4, H, W)'},
    {'step': 2, 'name': 'Normalize', 'description': 'Divide by 10000', 'formula': 'x_norm = x / 10000'},
    {'step': 3, 'name': 'Clip', 'description': 'Ensure [0, 1] range', 'formula': 'x_clipped = clip(x_norm, 0, 1)'},
    {'step': 4, 'name': 'Extract mask', 'description': 'Use target channel', 'values': [0,1,2,3]}
]

print("\nPreprocessing steps:")
for step in preprocessing_steps:
    print(f"\nStep {step['step']}: {step['name']}")
    print(f"  {step['description']}")
    if 'formula' in step:
        print(f"  {step['formula']}")

print(f"\n=== Data Value Ranges ===")
print("Before: Sentinel-2 L2A digital numbers: 0-10000")
print("After:  Normalized range: [0.0, 1.0]")

# Verify with actual data
sample = dataset.read(0)
image_path = sample.read(0)
with rasterio.open(image_path) as src:
    image = src.read()
    print(f"\nActual data (sample 0):")
    print(f"  Min: {image.min():.0f}, Max: {image.max():.0f}, Mean: {image.mean():.1f}, Std: {image.std():.1f}")
    print(f"  Data type: {image.dtype}")

## Step 9: Save Configuration Files

In [ ]:
# Save split information
split_info = {
    'full_dataset_size': int(full_size),
    'subset_fraction': float(subset_fraction),
    'subset_size': int(subset_size),
    'train_size': int(train_size),
    'val_size': int(val_size),
    'test_size': int(test_size),
    'train_indices': train_indices,
    'val_indices': val_indices,
    'test_indices': test_indices,
    'random_seed': 42
}

with open('../outputs/split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)
print("✓ Split info saved")

# Save preprocessing config
preprocessing_config = {
    'version': '1.0',
    'selected_bands': {
        'indices': [3, 2, 1, 7],
        'names': ['Red (B4)', 'Green (B3)', 'Blue (B2)', 'NIR (B8)']
    },
    'normalization': {
        'method': 'linear_scaling',
        'scale_factor': 10000,
        'output_range': [0, 1]
    },
    'target_format': {
        'type': 'multiclass_segmentation',
        'classes': class_labels,
        'num_classes': 4
    }
}

with open('../outputs/preprocessing_config.json', 'w') as f:
    json.dump(preprocessing_config, f, indent=2)
print("✓ Preprocessing config saved")

# Save class info
class_info = {
    'num_classes': 4,
    'classes': class_labels,
    'distribution': {str(k): {'count': int(v), 'percentage': float(v/total_pixels*100)} 
                    for k, v in sorted(class_distribution.items())}
}

with open('../outputs/class_info.json', 'w') as f:
    json.dump(class_info, f, indent=2)
print("✓ Class info saved")

print("\n✓ All configuration files saved to outputs/")

## Week 1 Summary

In [ ]:
print("\n" + "="*75)
print("WEEK 1 - CLOUDSEN12 DATASET EXPLORATION - COMPLETE")
print("="*75)

print(f"\n✓ DATASET SELECTION")
print(f"  Dataset: CloudSEN12 (Sentinel-2 L2A)")
print(f"  Total samples: {full_size:,}")
print(f"  Subset: {subset_size:,} ({subset_fraction*100:.1f}%)")

print(f"\n✓ DATA STRUCTURE")
print(f"  Format: GeoTIFF (via rasterio)")
print(f"  Bands: 12 total (11 spectral + 1 SCL)")
print(f"  Resolution: 10m or 20m per band")
print(f"  Pixel range: 0-10000 (digital numbers)")

print(f"\n✓ TARGET LABELS (4 classes)")
for val, label in sorted(class_labels.items()):
    pct = class_distribution[val] / total_pixels * 100
    print(f"  Class {val}: {label:20s} ({pct:5.2f}%)")

print(f"\n✓ INPUT BANDS SELECTED")
print(f"  Strategy: RGB + NIR (4 bands)")
print(f"  Indices: {selected_band_indices}")
print(f"  Names: {', '.join(selected_band_names)}")
print(f"  Output: (4, H, W) with values in [0, 1]")

print(f"\n✓ PREPROCESSING PIPELINE")
print(f"  1. Extract 4 selected bands")
print(f"  2. Normalize: x / 10000")
print(f"  3. Clip to [0, 1]")

print(f"\n✓ DATA SPLIT")
print(f"  Training:   {train_size:,d} samples (60%)")
print(f"  Validation: {val_size:,d} samples (20%)")
print(f"  Test:       {test_size:,d} samples (20%)")

print(f"\n✓ EVALUATION METRICS")
print(f"  Pixel-level: Precision, Recall, F1-score, IoU")
print(f"  Baseline: Sentinel-2 SCL")

print(f"\n✓ OUTPUT ARTIFACTS")
print(f"  Visualizations:")
print(f"    - outputs/figures/01_dataset_samples.png")
print(f"    - outputs/figures/02_class_distribution.png")
print(f"  Configuration:")
print(f"    - outputs/split_info.json")
print(f"    - outputs/preprocessing_config.json")
print(f"    - outputs/class_info.json")
print(f"  Python modules:")
print(f"    - src/preprocessing.py")
print(f"    - src/config.py")
print(f"  Documentation:")
print(f"    - WEEK1_REPORT.md")

print(f"\n✓ NEXT STEPS (Week 2)")
print(f"  → Implement Random Forest classifier")
print(f"  → Extract Sentinel-2 SCL baseline")
print(f"  → Evaluate and compare models")

print("\n" + "="*75)
print("✓ WEEK 1 COMPLETE - Ready for Week 2")
print("="*75 + "\n")